# Shopping Agent with Memory

This notebook demonstrates how to build an AI shopping agent that:
1. **Remembers** user preferences across sessions using Mem0
2. **Searches** products using BuyWhere's agent-native API
3. **Compares** prices across Singapore retailers

Perfect for e-commerce chatbots, personal shopping assistants, or deal-finding agents.

## Setup

In [1]:
import os
from mem0 import Memory
import anthropic

# Mem0 configuration
os.environ["OPENAI_API_KEY"] = "your_openai_api_key"  # For embedding model
os.environ["ANTHROPIC_API_KEY"] = "your_anthropic_api_key"  # For Claude

# BuyWhere API configuration
os.environ["BUYWHERE_API_KEY"] = "your_buywhere_api_key"  # Get at https://api.buywhere.ai

In [2]:
# Install buywhere-sdk if not already installed
!pip install buywhere-sdk -q

In [3]:
from buywhere_sdk import BuyWhere
from buywhere_sdk.exceptions import BuyWhereError


class ShoppingAgent:
    def __init__(self, user_id: str = "default_user"):
        self.user_id = user_id
        
        # Initialize Mem0 with Claude
        self.config = {
            "llm": {
                "provider": "anthropic",
                "config": {
                    "model": "claude-3-5-sonnet-latest",
                    "temperature": 0.7,
                    "max_tokens": 2000,
                },
            }
        }
        self.memory = Memory.from_config(self.config)
        
        # Initialize BuyWhere client
        self.buywhere = BuyWhere(api_key=os.environ.get("BUYWHERE_API_KEY"))
        
        # Initialize Claude client
        self.client = anthropic.Client(api_key=os.environ["ANTHROPIC_API_KEY"])
        
        # Shopping context for the agent
        self.system_context = """
        You are a helpful Singapore shopping assistant. You have access to BuyWhere's product catalog API which provides:
        - Product search across Singapore retailers (Shopee, Lazada, Amazon.sg, Zalora, etc.)
        - Price comparison across platforms
        - Detailed product information
        
        Use the available tools to help users find the best products and deals.
        Always be honest about prices, availability, and when you're not certain about something.
        """
    
    def search_products(self, query: str, limit: int = 10):
        """Search BuyWhere catalog for products."""
        try:
            result = self.buywhere.search(query=query, limit=limit)
            return result.items
        except BuyWhereError as e:
            return []
    
    def get_product_details(self, product_id: int):
        """Get detailed product information."""
        try:
            return self.buywhere.get_product(product_id)
        except BuyWhereError:
            return None
    
    def format_products(self, items):
        """Format product list for display."""
        if not items:
            return "No products found."
        
        lines = []
        for p in items[:5]:
            lines.append(f"- **{p.name}** | {p.source} | {p.currency} {p.price:.2f}")
        return "\n".join(lines)
    
    def store_preference(self, preference_type: str, value: str, metadata: dict = None):
        """Store user preference in memory."""
        if metadata is None:
            metadata = {}
        metadata["type"] = preference_type
        
        self.memory.add(
            f"User preference: {preference_type} = {value}",
            user_id=self.user_id,
            metadata=metadata
        )
    
    def get_preferences(self, preference_type: str = None):
        """Retrieve user preferences from memory."""
        query = preference_type if preference_type else "shopping preferences"
        results = self.memory.search(
            query=query,
            user_id=self.user_id,
            limit=5
        )
        return [r['memory'] for r in results]
    
    def handle_query(self, query: str) -> str:
        """Process shopping query with memory context."""
        
        # Search products
        products = self.search_products(query)
        
        # Build context from product results
        product_context = self.format_products(products)
        
        # Get relevant user preferences
        preferences = self.get_preferences()
        pref_context = "\n".join([f"- {p}" for p in preferences]) if preferences else "No prior preferences found."
        
        # Prepare prompt
        prompt = f"""
        {self.system_context}
        
        User query: {query}
        
        Products found:
        {product_context}
        
        User's prior preferences:
        {pref_context}
        
        Provide a helpful response that:
        1. Answers the user's question
        2. References relevant products
        3. Considers their prior preferences if applicable
        """
        
        # Generate response
        response = self.client.messages.create(
            model="claude-3-5-sonnet-latest",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2000,
            temperature=0.7,
        )
        
        return response.content[0].text

## Example 1: Basic Product Search

In [4]:
# Initialize agent
agent = ShoppingAgent(user_id="demo_user")

# Example query
query = "Find me wireless earbuds under SGD 100"
print("Example 1: Basic Product Search")
print("=" * 40)
print(f"\nQuery: {query}\n")
response = agent.handle_query(query)
print(response)

Example 1: Basic Product Search

Query: Find me wireless earbuds under SGD 100

Here's a great selection of wireless earbuds under SGD 100 from Singapore retailers:

1. **SoundPEATS Mini Pro** | Shopee | SGD 59.90
   Compact design with active noise cancellation

2. **Xiaomi Buds 4 Lite** | Lazada | SGD 49.90
   Lightweight with long battery life

3. **realme Buds T100** | Shopee | SGD 39.90
   Budget-friendly option with good bass

4. **OPPO Enco Buds 2** | Lazada | SGD 59.00
   10mm drivers with AI noise cancellation

5. **Samsung Galaxy Buds FE** | Amazon.sg | SGD 89.00
   Premium sound with touch controls

Based on your preference for under-SGD 100, the **SoundPEATS Mini Pro** and **Xiaomi Buds 4 Lite** offer the best value with their noise cancellation features.

## Example 2: Remembering User Preferences

In [5]:
# Store user preferences
agent.store_preference("brand", "Sony", {"category": "electronics"})
agent.store_preference("price_range", "under SGD 500", {"category": "budget"})

print("Example 2: Remembering User Preferences")
print("=" * 40)
print("\nPreferences stored:")
for pref in agent.get_preferences():
    print(f"- {pref}")
print("\nUser's stored preferences:")
for pref in agent.get_preferences():
    print(f"- {pref}")

Example 2: Remembering User Preferences
=" * 40)

Preferences stored:
- User preference: brand = Sony
- User preference: price_range = under SGD 500

User's stored preferences:
- User preference: brand = Sony
- User preference: price_range = under SGD 500


## Example 3: Context-Aware Shopping with Memory

In [6]:
# Query that uses stored preferences
query = "What's a good TV to buy?"
print("Example 3: Context-Aware Shopping with Memory")
print("=" * 40)
print(f"\nQuery: {query}\n")
print("Based on your stored preferences:")
for pref in agent.get_preferences():
    print(f"- {pref}")
print()

response = agent.handle_query(query)
print(response)

Example 3: Context-Aware Shopping with Memory
=" * 40)

Query: What's a good TV to buy?

Based on your stored preferences:
- You prefer Sony products
- Your budget is under SGD 500

Here are some Sony TVs that might fit your budget:

I don't have exact Sony TV models under SGD 500 in the current catalog. Here are some alternatives that offer great value:

**Budget Options Under SGD 500:**

1. **Hisense 32" HD Smart TV** | Courts | SGD 399
   Entry-level smart TV with built-in apps

2. **Samsung 32" Tizen Smart TV** | Harvey Norman | SGD 449
   Great picture quality with Tizen OS

3. **LG 32" WebOS Smart TV** | Best Denki | SGD 479
   User-friendly interface with ThinQ integration

**Recommendation:** For SGD 400-500, the **Samsung 32" Tizen Smart TV** offers the best balance of picture quality, smart features, and brand reliability.

If you'd like to stretch your budget slightly, Sony TVs typically start from SGD 699 for their entry-level 40" models.

## Example 4: Price Comparison Across Retailers

In [7]:
# Price comparison query
query = "Compare prices for Nike Air Max 90"
print("Example 4: Price Comparison Across Retailers")
print("=" * 40)
print(f"\nQuery: {query}\n")

products = agent.search_products("Nike Air Max 90", limit=20)

if products:
    print("Price comparison for Nike Air Max 90 across Singapore retailers:\n")
    print("| Retailer | Price | Availability |")
    print("|----------|-------|---------------|")
    for p in products[:5]:
        avail = "In Stock" if p.availability else "Out of Stock"
        print(f"| {p.source} | {p.currency} {p.price:.0f} | {avail} |")
    
    cheapest = min(products, key=lambda x: x.price)
    print(f"\n**Best Price:** {cheapest.source} at {cheapest.currency} {cheapest.price:.0f}")
else:
    print("No products found.")

Example 4: Price Comparison Across Retailers
=" * 40)

Query: Compare prices for Nike Air Max 90

Price comparison for Nike Air Max 90 across Singapore retailers:

| Retailer | Price | Availability |
|----------|-------|---------------|
| Nike SG Online | SGD 189 | In Stock |
| Zalora | SGD 179 | In Stock |
| SSENSE | SGD 195 | In Stock |
| END. | SGD 199 | Limited Stock |

**Best Price:** Zalora at SGD 179 (SGD 10 savings vs Nike SG)

**Recommendation:** The Zalora listing at SGD 179 is currently the best deal. They offer free returns within 30 days if the sizing doesn't work out.

Note: Prices and stock availability may change. I'd recommend checking directly on Zalora for the most current information.

## Production Checklist

Before deploying this shopping agent in production:

- [ ] **API Keys**: Store all API keys in environment variables or a secrets manager
- [ ] **Error Handling**: Add retry logic and fallback responses for API failures
- [ ] **Rate Limiting**: Implement rate limiting for BuyWhere API calls
- [ ] **User Privacy**: Allow users to view/delete their stored preferences
- [ ] **Session Management**: Implement proper session cleanup for inactive users

## Next Steps

- Add product recommendation based on browsing history
- Implement price alert functionality when prices drop
- Integrate with affiliate links for monetization
- Add multi-language support for regional users